In [1]:
import pandas as pd

df = pd.read_csv("yellow_tripdata_2021-01.csv.gz", nrows=100)
print(df.head())
print(df.dtypes)

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10           1                  N           142            43   
1           0.20           1                  N           238           151   
2          14.70           1                  N           132           165   
3          10.60           1                  N           138           132   
4           4.94           1                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  \


In [2]:
import pandas as pd

path = "yellow_tripdata_2021-01.csv.gz"
total = 0
nulls = None
mins = None
maxs = None
dtypes = None

for chunk in pd.read_csv(path, chunksize=100_000):
    total += len(chunk)
    n = chunk.isna().sum()
    nulls = n if nulls is None else nulls + n
    mn = chunk.min(numeric_only=False)
    mx = chunk.max(numeric_only=False)
    mins = mn if mins is None else pd.concat([mins, mn], axis=1).min(axis=1)
    maxs = mx if maxs is None else pd.concat([maxs, mx], axis=1).max(axis=1)
    dtypes = chunk.dtypes

profile = pd.DataFrame({
    "dtype": dtypes.astype(str),
    "nulls": nulls,
    "nulls_pct": (nulls / total * 100).round(2),
    "min": mins,
    "max": maxs,
})
print("Total rows:", total)
print(profile)

/tmp/ipykernel_3508/2260815362.py:10: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(path, chunksize=100_000):


TypeError: '<=' not supported between instances of 'str' and 'float'

In [3]:
import pandas as pd

path = "yellow_tripdata_2021-01.csv.gz"
date_cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime"]

total = 0
nulls = None
mins, maxs = {}, {}
dtypes = None

for chunk in pd.read_csv(path, chunksize=100_000, dtype={"store_and_fwd_flag": "string"}):
    total += len(chunk)
    n = chunk.isna().sum()
    nulls = n if nulls is None else nulls + n
    dtypes = chunk.dtypes

    for c in chunk.columns:
        if c == "store_and_fwd_flag":
            continue
        s = chunk[c]
        if c in date_cols:
            s = s.dropna()   # dates stay as text; ISO format sorts correctly
        if s.dropna().empty:
            continue
        mn, mx = s.min(), s.max()
        mins[c] = mn if c not in mins else min(mins[c], mn)
        maxs[c] = mx if c not in maxs else max(maxs[c], mx)

profile = pd.DataFrame({
    "dtype": dtypes.astype(str),
    "nulls": nulls,
    "nulls_pct": (nulls / total * 100).round(2),
    "min": pd.Series(mins),
    "max": pd.Series(maxs),
})
print("Total rows:", total)
print(profile)

Total rows: 1369765
                         dtype  nulls  nulls_pct                  min  \
DOLocationID             int64      0       0.00                    1   
PULocationID             int64      0       0.00                    1   
RatecodeID             float64  98352       7.18                    1   
VendorID               float64  98352       7.18                    1   
congestion_surcharge   float64      0       0.00                 -2.5   
extra                  float64      0       0.00                 -5.5   
fare_amount            float64      0       0.00               -490.0   
improvement_surcharge  float64      0       0.00                 -0.3   
mta_tax                float64      0       0.00                 -0.5   
passenger_count        float64  98352       7.18                    0   
payment_type           float64  98352       7.18                    1   
store_and_fwd_flag      string  98352       7.18                  NaN   
tip_amount             float64 

In [4]:
import pandas as pd

path = "yellow_tripdata_2021-01.csv.gz"
cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
        "trip_distance", "fare_amount", "total_amount"]

counts = {
    "pickup_outside_jan_2021": 0,
    "dropoff_before_pickup": 0,
    "fare_negative": 0,
    "fare_zero": 0,
    "distance_negative": 0,
    "distance_zero": 0,
    "passenger_count_zero": 0,
    "passenger_count_null": 0,
}
total = 0

for chunk in pd.read_csv(path, chunksize=100_000, usecols=cols):
    total += len(chunk)
    pu = pd.to_datetime(chunk["tpep_pickup_datetime"], errors="coerce")
    do = pd.to_datetime(chunk["tpep_dropoff_datetime"], errors="coerce")

    counts["pickup_outside_jan_2021"] += int(((pu < "2021-01-01") | (pu >= "2021-02-01")).sum())
    counts["dropoff_before_pickup"] += int((do < pu).sum())
    counts["fare_negative"] += int((chunk["fare_amount"] < 0).sum())
    counts["fare_zero"] += int((chunk["fare_amount"] == 0).sum())
    counts["distance_negative"] += int((chunk["trip_distance"] < 0).sum())
    counts["distance_zero"] += int((chunk["trip_distance"] == 0).sum())
    counts["passenger_count_zero"] += int((chunk["passenger_count"] == 0).sum())
    counts["passenger_count_null"] += int(chunk["passenger_count"].isna().sum())

problems = pd.DataFrame({
    "problem": list(counts.keys()),
    "rows": list(counts.values()),
})
problems["pct"] = (problems["rows"] / total * 100).round(3)
print("Total rows:", total)
print(problems)

Total rows: 1369765
                   problem   rows    pct
0  pickup_outside_jan_2021     24  0.002
1    dropoff_before_pickup   5642  0.412
2            fare_negative   6769  0.494
3                fare_zero    642  0.047
4        distance_negative      0  0.000
5            distance_zero  19952  1.457
6     passenger_count_zero  26726  1.951
7     passenger_count_null  98352  7.180


In [5]:
import pandas as pd

path = "yellow_tripdata_2021-01.csv.gz"
seen = set()
total = 0
dupes = 0

for chunk in pd.read_csv(path, chunksize=100_000, dtype={"store_and_fwd_flag": "string"}):
    total += len(chunk)
    hashes = pd.util.hash_pandas_object(chunk, index=False)
    for h in hashes:
        if h in seen:
            dupes += 1
        else:
            seen.add(h)

dup_table = pd.DataFrame({
    "problem": ["duplicate_rows"],
    "rows": [dupes],
    "pct": [round(dupes / total * 100, 3)],
})
print("Total rows:", total)
print(dup_table)

Total rows: 1369765
          problem  rows  pct
0  duplicate_rows     0  0.0


In [6]:
import pandas as pd

def clean(df):
    """Returns (cleaned_df, {rule: rows_removed}). Original df is not changed."""
    df = df.copy()
    removed = {}

    pu = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
    do = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")
    df["tpep_pickup_datetime"] = pu
    df["tpep_dropoff_datetime"] = do

    def drop(mask, name):
        nonlocal df
        removed[name] = int(mask.sum())
        df = df[~mask]

    drop((df["tpep_pickup_datetime"] < "2021-01-01") |
         (df["tpep_pickup_datetime"] >= "2021-02-01") |
         df["tpep_pickup_datetime"].isna(), "pickup_outside_jan_2021")
    drop(df["tpep_dropoff_datetime"] < df["tpep_pickup_datetime"], "dropoff_before_pickup")
    drop(df["fare_amount"] < 0, "fare_negative")

    # flags: rows are kept, a column marks them
    df["flag_fare_zero"] = df["fare_amount"] == 0
    df["flag_distance_zero"] = df["trip_distance"] == 0
    df["flag_passenger_zero"] = df["passenger_count"] == 0
    df["flag_passenger_missing"] = df["passenger_count"].isna()

    return df, removed

In [7]:
path = "yellow_tripdata_2021-01.csv.gz"
before = after = 0
removed_total = {}
flag_total = {}

for chunk in pd.read_csv(path, chunksize=100_000, dtype={"store_and_fwd_flag": "string"}):
    before += len(chunk)
    cleaned, removed = clean(chunk)
    after += len(cleaned)
    for k, v in removed.items():
        removed_total[k] = removed_total.get(k, 0) + v
    for c in [c for c in cleaned.columns if c.startswith("flag_")]:
        flag_total[c] = flag_total.get(c, 0) + int(cleaned[c].sum())

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)
print(pd.DataFrame({"rule": removed_total.keys(), "rows_removed": removed_total.values()}))
print(pd.DataFrame({"flag": flag_total.keys(), "rows_flagged": flag_total.values()}))

Rows before: 1369765
Rows after: 1357332
Rows removed: 12433
                      rule  rows_removed
0  pickup_outside_jan_2021            24
1    dropoff_before_pickup          5642
2            fare_negative          6767
                     flag  rows_flagged
0          flag_fare_zero           627
1      flag_distance_zero         19308
2     flag_passenger_zero         26726
3  flag_passenger_missing         92600
